# Part d: L1 and L2 regularization


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax

from src.data import make_runge_split, make_runge_split_3, design_matrix, scale_data, center_data, runge
from src.functions import mse, mse_der, sigmoid, sigmoid_der, identity, identity_der
from src.network import NeuralNetwork
from src.optimizers import Adam
from src.regression import ols_fit, ridge_fit, lasso_fit
from src.metrics import r2
from src.plotting import line_plot, heatmap, save_fig

Dataset and best result from project 1: 
-n = 100 points
-sigma = 0.1 noise
-seed = 2026
-test size = 0.2
-Optimization method = Adam

In [ ]:
# recreating ridge and lasso results from project 1
# Use the same dataset and the best results from project 1.

degree = 8          # best Ridge
num_iters = 20000 
lam_ols = 0.0
lam_ridge = 4e-2    # best Ridge
lam_lasso = 1e-2    # used for Lasso
gamma = 0.01
seed = 2026

x_train, x_test, y_train, y_test = make_runge_split()                   # split 80/20
x_trn, x_val, x_tst, y_trn, y_val, y_tst = make_runge_split_3()         # 64/16/20, for tuning
assert np.array_equal(x_tst, x_test) and np.array_equal(y_tst, y_test)  # same test set in both splits

print("train", x_train.shape, " test", x_test.shape)
print("tuning: train", x_trn.shape, " validation", x_val.shape)

In [ ]:
# design matrix
matrix_train = design_matrix(x_train, degree, intercept=False)
matrix_test = design_matrix(x_test, degree, intercept=False)
P_train, P_test, P_mean, P_std = scale_data(matrix_train, matrix_test)
y_train_c, y_test_c, y_mean = center_data(y_train, y_test)

def poly_pred(theta, x):
    P = (design_matrix(x, degree, intercept=False) - P_mean) / P_std
    return P @ theta + y_mean

# regression fits
theta_ols = ols_fit(P_train, y_train_c)
theta_ridge = ridge_fit(P_train, y_train_c, lam_ridge)
theta_lasso = lasso_fit(P_train, y_train_c, lam_lasso, gamma = 0.01, n_iter = 20000)

# predictions
results = {}
for name, theta in [(f'OLS, deg = {degree}', theta_ols),
                    (f'Ridge, deg = {degree}, lam = {lam_ridge}', theta_ridge),
                    (f'Lasso, deg = {degree}, lam = {lam_lasso}', theta_lasso)]:
    y_pred = poly_pred(theta, x_test)
    MSE = mse(y_pred, y_test)
    R2 = r2(y_pred, y_test)
    results[name] = (float(MSE), R2)
    print(f'{name:28s} | test MSE {results[name][0]:.4f} | R2 {results[name][1]:.3f}')

### Using 1D Runge function

In [ ]:
X_trn_s, X_val_s, x_mean_trn, x_std_trn = scale_data(x_trn, x_val) # scale with 64 tuning-training points
epochs = 500
batch_size = 14

def make_net(lam, reg):
    return NeuralNetwork(network_input_size = 1,
                        layer_output_sizes = [50 ,1],
                        activation_funcs = [sigmoid, identity], 
                        activation_ders = [sigmoid_der, identity_der],
                        cost_fun = mse,
                        cost_der = mse_der,
                        optimizer = Adam(gamma),
                        regularization = reg,
                        lmbda = lam,
                        seed = seed,
                        penalize_bias=False)

def train_net(X_fit, y_fit, X_eval, y_eval, reg, lam):
    neural_net = make_net(lam, reg)
    neural_net.train(X_fit, y_fit, epochs=epochs, batch_size=batch_size, rng = np.random.default_rng(seed))
    MSE = mse(neural_net.predict(X_eval), y_eval)
    score = float(MSE)
    return neural_net, (score if np.isfinite(score) else np.nan)


### Add hyperparameters $\lambda$ with the $L_2$ and $L_1$ norms (sec 8.12).

In [ ]:
# Adding L1 to network
L1_net = make_net(lam = lam_lasso, reg = 'L1')

for l in [0.0, 1e-4, 1e-2, 1]:
    _, val = train_net(X_trn_s, y_trn, X_val_s, y_val, reg = 'L1', lam = lam_lasso)
    print(f'L1: lam = {l:6g}: validation MSE {val:.4f}')

In [ ]:
# Adding L2 to network
L2_net = make_net(reg = 'L2', lam = lam_ridge)

for l in [0.0, 1e-4, 1e-2, 1]:
    _, val = train_net(X_trn_s, y_trn, X_val_s, y_val, reg = 'L2', lam = lam_ridge)
    print(f'L2: lam = {l:6g}: validation MSE {val:.4f}')

In [ ]:
# Find the optimal results for the hyperparameters $\lambda$ and the learning rates $\gamma$ and neural network - architecture.

In [ ]:
# Compare the $L_2$ results with Ridge regression from project 1.
# Use the same dataset and the best results from project 1.

In [ ]:
# Compare the $L_1$ results with the Lasso calculations of project 1.
# Use the same dataset and the best results from project 1.